### import yfinance for accessing financial data
### import pandas for data analysis
### import nump

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

spy = yf.download(
    "SPY",
    start = "2000-01-03",
    end = "2026-09-20",
    auto_adjust = False
)

spy.head()

[*********************100%***********************]  1 of 1 completed


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY,SPY
Date,,,,,,
2000-01-03,90.907005,145.4375,148.25000,143.875000,148.25000,8164300
2000-01-04,87.351990,139.7500,144.06250,139.640625,143.53125,8089800
2000-01-05,87.508240,140.0000,141.53125,137.250000,139.93750,12177900
2000-01-06,86.101883,137.7500,141.50000,137.750000,139.62500,6227200
2000-01-07,91.102356,145.7500,145.75000,140.062500,140.31250,8066500


### Construct log returns. Log_return_t = ln(price_t/price_{t+1})

In [2]:
spy['log_returns'] = np.log(spy['Close']/spy['Close'].shift())
spy[['Close','log_returns']]

Price,Close,log_returns
Ticker,SPY,
Date,,
2000-01-03,145.437500,NaN
2000-01-04,139.750000,-0.039891
2000-01-05,140.000000,0.001787
2000-01-06,137.750000,-0.016202
2000-01-07,145.750000,0.056452
...,...,...
2026-09-14,760.880005,-0.004472
2026-09-15,757.390015,-0.004597


In [3]:
print(spy.columns)

MultiIndex([(  'Adj Close', 'SPY'),
            (      'Close', 'SPY'),
            (       'High', 'SPY'),
            (        'Low', 'SPY'),
            (       'Open', 'SPY'),
            (     'Volume', 'SPY'),
            ('log_returns',    '')],
           names=['Price', 'Ticker'])


### Columns are multi Index, Let me flatten them

In [4]:
spy.columns = [
    col[0] if col[1] == '' else col[0]
    for col in spy.columns
]
print(spy.columns)

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume', 'log_returns'], dtype='str')


### Analyse the statistics of log_returns

In [5]:
spy['log_returns'].describe()

count    6717.000000
mean        0.000247
std         0.012157
min        -0.115887
25%        -0.004784
50%         0.000615
75%         0.005953
max         0.135577
Name: log_returns, dtype: float64

In [6]:
spy['log_returns'].mean()/spy['log_returns'].std()

np.float64(0.020276835843136356)

### Predicting the future using the past(Lags)

In [7]:
spy['log_returns_lag_1'] = spy['log_returns'].shift()
spy['log_returns_lag_2'] = spy['log_returns'].shift(2)
spy[['log_returns', 'log_returns_lag_1','log_returns_lag_2']]


,log_returns,log_returns_lag_1,log_returns_lag_2
Date,,,
2000-01-03,NaN,NaN,NaN
2000-01-04,-0.039891,NaN,NaN
2000-01-05,0.001787,-0.039891,NaN
2000-01-06,-0.016202,0.001787,-0.039891
2000-01-07,0.056452,-0.016202,0.001787
...,...,...,...
2026-09-14,-0.004472,0.008488,-0.006012
2026-09-15,-0.004597,-0.004472,0.008488
2026-09-16,-0.004420,-0.004597,-0.004472


### Correlation of lags.

In [8]:
spy[['log_returns', 'log_returns_lag_1','log_returns_lag_2']].dropna().corr()

,log_returns,log_returns_lag_1,log_returns_lag_2
log_returns,1.000000,-0.084651,-0.017378
log_returns_lag_1,-0.084651,1.000000,-0.084634
log_returns_lag_2,-0.017378,-0.084634,1.000000


### Hypothesis: From correlation data, probably Mean inversion? negative but very close to zero, maybe not correlated that much.

### We now develop the Statistical Edge

In [9]:
spy_arr = spy.dropna(subset = [
    'log_returns',
    'log_returns_lag_1',
    'log_returns_lag_2'
])
spy_arr

,Adj Close,Close,High,Low,Open,Volume,log_returns,log_returns_lag_1,log_returns_lag_2
Date,,,,,,,,,
2000-01-06,86.101883,137.750000,141.500000,137.750000,139.625000,6227200,-0.016202,0.001787,-0.039891
2000-01-07,91.102356,145.750000,145.750000,140.062500,140.312500,8066500,0.056452,-0.016202,0.001787
2000-01-10,91.414818,146.250000,146.906250,145.031250,146.250000,5741700,0.003425,0.056452,-0.016202
2000-01-11,90.321014,144.500000,146.093750,143.500000,145.812500,7503700,-0.012038,0.003425,0.056452
2000-01-12,89.422508,143.062500,144.593750,142.875000,144.593750,6907700,-0.009998,-0.012038,0.003425
...,...,...,...,...,...,...,...,...,...
2026-09-14,758.995239,760.880005,763.520020,757.929993,759.000000,43992400,-0.004472,0.008488,-0.006012
2026-09-15,755.513916,757.390015,760.349976,756.150024,760.119995,46195000,-0.004597,-0.004472,0.008488
2026-09-16,752.182190,754.049988,761.669983,749.599976,759.500000,59217700,-0.004420,-0.004597,-0.004472


In [10]:
y = spy_arr['log_returns'].values
y

array([-0.01620197,  0.05645238,  0.00342466, ..., -0.00441967,
        0.01127495, -0.00119396], shape=(6715,))

In [11]:
X = spy_arr[['log_returns_lag_1', 'log_returns_lag_2']].values
X

array([[ 0.00178731, -0.03989133],
       [-0.01620197,  0.00178731],
       [ 0.05645238, -0.01620197],
       ...,
       [-0.00459733, -0.0044716 ],
       [-0.00441967, -0.00459733],
       [ 0.01127495, -0.00441967]], shape=(6715, 2))

In [12]:
y.shape

(6715,)

In [13]:
X.shape

(6715, 2)

### Split testing with training data.

In [14]:
def time_split(x,train_size = 0.75):
    i = int(len(x)* train_size)
    return x[:i].copy(), x[i:].copy()
spy_train, spy_test = time_split(spy_arr,train_size=0.5)

### Train the model(Akusetswenzwe!!)

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim

# Input features(Model predictors)
features = ['log_returns_lag_1','log_returns_lag_2']
# Target variable(Model output)
target = 'log_returns'

# convert train/test splits into PyTorch tensors.
X_train = torch.tensor(spy_train[features].values, dtype=torch.float32)
X_test = torch.tensor(spy_test[features].values,dtype=torch.float32)

#Create target tensors and add a column dimension(N->N+1)
y_train = torch.tensor(spy_train[target].values, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(spy_test[target].values,dtype=torch.float32).unsqueeze(1)

#DEFINING THE MODEL(MAKHWAPHENI)
#no of input features
no_features = len(features)

#simple linear regression model regression model
makhwapheni = nn.Linear(no_features, 1) # makhwapheni = B_0 + B_1x_1 + B_2x_2

# HuberLoss, error between prediction and actual values
criterion = nn.HuberLoss()

#Stochastic gradient Descent optimizer
optimizer = optim.SGD(makhwapheni.parameters(), lr=0.01)

# TRAINING LOOP(FULL-BATCH GRADIENT DESCENT)
for epoch in range(5000):
    #clear previously stored gradients(They accumulate by default)
    optimizer.zero_grad()

    #Forward pass:compute predictions
    y_pred = makhwapheni(X_train)

    #Compute loss between predictions and true values
    loss = criterion(y_pred,y_train)

    #BackPropergation: Compute gradients of loss w.r.t parameters
    loss.backward()

    #Update model parameters using the combined gradients
    optimizer.step()

    #Print loss every 500 epochs
    if epoch % 500 == 0:
        print("Epoch:", epoch, "loss:", loss.item())

#CHECK TRAINED PARAMETERS
print("Final weight:", makhwapheni.weight.data)
print("Final bias:", makhwapheni.bias.data)




Epoch: 0 loss: 0.1644880771636963
Epoch: 500 loss: 0.00015740562230348587
Epoch: 1000 loss: 0.00015019631246104836
Epoch: 1500 loss: 0.0001500811631558463
Epoch: 2000 loss: 0.00014996643585618585
Epoch: 2500 loss: 0.000149851810419932
Epoch: 3000 loss: 0.00014973728684708476
Epoch: 3500 loss: 0.00014962287968955934
Epoch: 4000 loss: 0.00014950858894735575
Epoch: 4500 loss: 0.00014939438551664352
Final weight: tensor([[-0.2990,  0.6932]])
Final bias: tensor([3.6836e-05])


## Actually building the edge

In [16]:
w,b = makhwapheni.weight, makhwapheni.bias.item()
w

Parameter containing:
tensor([[-0.2990,  0.6932]], requires_grad=True)

In [17]:
#Save the model
torch.save(makhwapheni.state_dict(), "makhwapheni.pth")

In [18]:
#Evaluate Model
y_hat = makhwapheni(X_test)
y_hat

tensor([[ 0.0021],
        [-0.0025],
        [ 0.0056],
        ...,
        [-0.0017],
        [-0.0018],
        [-0.0064]], grad_fn=<AddmmBackward0>)

In [19]:
#remove grad() mess
y_hat = y_hat.detach().squeeze()
y_hat

tensor([ 0.0021, -0.0025,  0.0056,  ..., -0.0017, -0.0018, -0.0064])

In [20]:
#Add to DataFrame
spy_test['y_hat'] = y_hat
spy_test[['log_returns', 'y_hat']]

,log_returns,y_hat
Date,,
2013-05-14,0.010281,0.002051
2013-05-15,0.005372,-0.002486
2013-05-16,-0.004706,0.005557
2013-05-17,0.009631,0.005168
2013-05-20,-0.000060,-0.006105
...,...,...
2026-09-14,-0.004472,-0.006668
2026-09-15,-0.004597,0.007257
2026-09-16,-0.004420,-0.001688


In [21]:
#Add directional signal
spy_test['dir_signal'] = np.sign(spy_test['y_hat'])

In [22]:
#Evaluate directional signal
spy_test['is_won'] = spy_test['dir_signal'] == np.sign(spy_test[target])
spy_test[['log_returns', 'y_hat', 'is_won']]

,log_returns,y_hat,is_won
Date,,,
2013-05-14,0.010281,0.002051,True
2013-05-15,0.005372,-0.002486,False
2013-05-16,-0.004706,0.005557,False
2013-05-17,0.009631,0.005168,True
2013-05-20,-0.000060,-0.006105,True
...,...,...,...
2026-09-14,-0.004472,-0.006668,True
2026-09-15,-0.004597,0.007257,False
2026-09-16,-0.004420,-0.001688,True


In [23]:
#Determine Directional Accuracy(DA)
da = spy_test['is_won'].mean()
da

np.float64(0.5128052412150089)

In [24]:
#Add Trade return(positive if won, negative if lost)
spy_test['trade_log_return'] = spy_test['dir_signal'] * spy_test[target]

In [25]:
#EVALUATE STATISTICAL EDGE
has_statistical_edge = spy_test['trade_log_return'].mean() > 0
has_statistical_edge

np.True_

In [26]:
#EVALUATE TOTAL RETURN
sum_logs = spy_test['trade_log_return'].sum()
sum_returns = np.exp(sum_logs)
sum_returns

np.float64(2.1791056155457884)

In [27]:
#WVALUATE EQUITY CURVE
cum_trade_log_returns = spy_test['trade_log_return'].cumsum()
cum_trade_log_returns

Date
2013-05-14    0.010281
2013-05-15    0.004909
2013-05-16    0.000202
2013-05-17    0.009833
2013-05-20    0.009893
                ...   
2026-09-14    0.789173
2026-09-15    0.784576
2026-09-16    0.788996
2026-09-17    0.777721
2026-09-18    0.778915
Name: trade_log_return, Length: 3358, dtype: float64